# 04 — Spatial-scale sensitivity and forest-structure dependence

**Research aim:** When can SAM1 crown outputs provide reliable ecological forest metrics, and how does reliability depend on evaluation scale and structural complexity?

A complementary ecological analysis for Nadine's SAM-based crown segmentation manuscript. **No GPU, inference, training or model tuning.** The frozen SAM1 ViT-H NMS benchmark and Kristof's Baego reference polygons come from Crown Notebooks 00–03.

**Two linked questions**
1. Does canopy-cover agreement at ~2 ha conceal larger 10–80 m window errors, and do commission/omission errors cancel as evaluation area increases?
2. Are errors related to observed *reference* crown density, size, size heterogeneity and local crowding (rather than primarily Musanga)?

**Experimental guardrails:** Six Baego ROIs—not independent crown-level experimental replicates. Non-overlapping full square windows (10, 20, 40, 80 m); an 80 m window may not fit an irregular ROI and will be explicitly absent rather than silently extended. Evaluation windows are **not inference tiles**. Count crowns by centroid inside window, canopy as union of clipped footprints including SAM halo. Report both absolute and relative errors; zero-reference-count windows have undefined relative density error. A full-ROI comparison is included as a separate benchmark. Main analysis uses NMS; optional author-style contrast later. Source references are provisional and the SAM workflow differs from manuscript's final SAM-frst configuration.

**Outputs:** `V2/Crown/scale_structure_analysis/` CSVs and publication-oriented figures. Results need replication with manuscript's final SAM-frst polygons before quantitative claims about that workflow.

## 01 — Setup, data paths and frozen analysis settings

In [ ]:
# 01 — CPU setup; consistent inputs from Notebooks 00–03
!pip -q install geopandas shapely scipy rasterio

from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
import rasterio, matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon, box
from shapely.ops import unary_union
from shapely import make_valid
from scipy.spatial import cKDTree
from scipy.stats import spearmanr
from google.colab import drive

drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
DATA=ROOT/"data/Geo_Ref"
RGB=DATA/"Baego_georef.tif"
ROI_FILE=DATA/"Annotations/ROI_Crowns_Kvo.shp"
REF_FILE=DATA/"Annotations/Crown_Kristof.shp"
SAM_DIR=ROOT/"V2/Crown/sam1_core_centroid_halo40_v2"
OUT=ROOT/"V2/Crown/scale_structure_analysis"
ROIS=tuple(range(6))
METHOD="nms_060"
SCALES_M=(10,20,40,80)
REFERENCE_MIN_AREA=5.0
# Full square windows entirely inside each irregular ROI; no fake 80m samples
MIN_INSIDE_FRACTION=0.999
for path in (RGB,ROI_FILE,REF_FILE,SAM_DIR):
    if not path.exists():raise FileNotFoundError(path)
for roi in ROIS:
    if not (SAM_DIR/f"roi{roi}_{METHOD}.gpkg").is_file():
        raise FileNotFoundError(SAM_DIR/f"roi{roi}_{METHOD}.gpkg")
OUT.mkdir(parents=True,exist_ok=True)
with rasterio.open(RGB) as ds:
    CRS, BOUNDS=ds.crs,ds.bounds
assert CRS is not None and CRS.is_projected
print("Evaluation window sizes (m):",SCALES_M)
print("ROI total areas and methods are independent of the SAM 40m inference core.")
print("Output:",OUT)

## 02 — Projected polygons, reference QA and six-ROI inventory

Reference crowns ≥5 m². Crown centroids and predicted polygons share the Baego raster CRS. Missing reference .prj is assigned only if raw coordinates overlap the raster. The 40 m exterior inference halo is preserved for boundary-crossing canopy cover.

In [ ]:
# 02 — Geometry normalization; load once
def polygonal(g):
    if g is None or g.is_empty:return Polygon()
    if not g.is_valid:g=make_valid(g)
    if isinstance(g,(Polygon,MultiPolygon)):return g
    parts=[a for a in getattr(g,"geoms",[]) if isinstance(a,(Polygon,MultiPolygon)) and not a.is_empty]
    return unary_union(parts) if parts else Polygon()

def read_projected(path):
    g=gpd.read_file(path)
    g=g.loc[g.geometry.notna() & ~g.geometry.is_empty].copy().reset_index(drop=True)
    if g.crs is None:
        b,rb=g.total_bounds,tuple(BOUNDS)
        if not (b[2]>=rb[0] and b[0]<=rb[2] and b[3]>=rb[1] and b[1]<=rb[3]):
            raise ValueError(f"Missing CRS and non-overlapping XY coordinates: {path}")
        print("Assigning Baego raster CRS:",path.name)
        g=g.set_crs(CRS)
    else:g=g.to_crs(CRS)
    g["geometry"]=gpd.GeoSeries([polygonal(x) for x in g.geometry],index=g.index,crs=CRS)
    return g.loc[~g.geometry.is_empty].reset_index(drop=True)

roi_gdf=read_projected(ROI_FILE)
assert len(roi_gdf)==6
reference=read_projected(REF_FILE)
reference=reference.loc[reference.geometry.area>=REFERENCE_MIN_AREA].reset_index(drop=True)
predicted={i:read_projected(SAM_DIR/f"roi{i}_{METHOD}.gpkg") for i in ROIS}
print("Six Baego ROIs; reference crowns:",len(reference))
display(pd.DataFrame([
    {"ROI":i,"roi_ha":roi_gdf.geometry.iloc[i].area/1e4,
     "reference_centroids":int(reference.geometry.centroid.within(roi_gdf.geometry.iloc[i]).sum()),
     "predicted_centroids":int(predicted[i].geometry.centroid.within(roi_gdf.geometry.iloc[i]).sum())}
    for i in ROIS]))

## 03 — Fixed, non-overlapping evaluation windows and coverage audit

Generate **full** square windows per ROI. Grid boundaries are anchored to a fixed global metric grid (`floor(x / size) * size`) rather than the SAM inference core grid. Retain only cells with ≥99.9% geometric intersection with the ROI. This avoids confusing a small clipped fragment with a true 40 × 40 m sampling unit. It may exclude edge areas; the whole-ROI result is also evaluated separately.

Window counts decline with scale. **If 80 m windows are unavailable, report this rather than inventing larger coverage**. Windows at different scales are nested/correlated, not independent replicates. Compare scale summaries using ROIs as the replication units, not individual window counts.

In [ ]:
# 03 — Non-overlapping squares per scale; measure sampling footprint
def windows_for_region(region,scale):
    xmin,ymin,xmax,ymax=region.bounds
    xs=range(int(np.floor(xmin/scale)),int(np.ceil(xmax/scale)))
    ys=range(int(np.floor(ymin/scale)),int(np.ceil(ymax/scale)))
    cells=[]
    for ix in xs:
        for iy in ys:
            square=box(ix*scale,iy*scale,(ix+1)*scale,(iy+1)*scale)
            fraction=square.intersection(region).area/(scale*scale)
            if fraction >= MIN_INSIDE_FRACTION:
                cells.append(square)
    return cells

WINDOWS=[]
for roi in ROIS:
    region=roi_gdf.geometry.iloc[roi]
    for scale in SCALES_M:
        squares=windows_for_region(region,scale)
        for j,square in enumerate(squares):
            WINDOWS.append({"ROI":roi,"scale_m":scale,"window_id":j,
                            "area_m2":square.area,"geometry":square})
WINDOWS_GDF=gpd.GeoDataFrame(WINDOWS,geometry="geometry",crs=CRS)
footprint=(WINDOWS_GDF.groupby(["ROI","scale_m"]).agg(
    n_windows=("window_id","size"),area_m2=("area_m2","sum")).reset_index())
allkeys=pd.MultiIndex.from_product([ROIS,SCALES_M],names=["ROI","scale_m"]).to_frame(index=False)
footprint=allkeys.merge(footprint,on=["ROI","scale_m"],how="left").fillna({"n_windows":0,"area_m2":0})
footprint["roi_ha"]=footprint.ROI.map({i:roi_gdf.geometry.iloc[i].area/1e4 for i in ROIS})
footprint["fraction_roi_sampled"]=footprint.area_m2/(footprint.roi_ha*10000)
print("=== Evaluation geometry coverage; missing 80m cells remain explicitly zero ===")
display(footprint.round(3))
footprint.to_csv(OUT/"04_window_coverage.csv",index=False)

## 04 — Ecological error at each window scale

For every window, calculate reference and SAM centroid counts, density, median/mean crown area, clipped union canopy cover and area IoU. Compute **signed cover bias (percentage points)**, **absolute cover error**, **reference-only** and **SAM-only** area percentages, and their sum (total spatial disagreement).

A key diagnostic is the **cancellation ratio**: how much of the gross omission-plus-commission disagreement disappears when only net cover bias is reported. Canopy IoU uses exact vector polygons.

Crown counts can be zero in small windows: preserve count errors, but do not compute undefined relative density errors. Median crown area requires enough reference crowns for interpretation.

In [ ]:
# 04 — Accurate vector area and crown metrics, indexed by spatial window
def union_clipped(geoms, region):
    clipped=[polygonal(g.intersection(region)) for g in geoms if g.intersects(region)]
    clipped=[g for g in clipped if not g.is_empty and g.area>0]
    return polygonal(unary_union(clipped)) if clipped else Polygon()

def metric_pair(roi_id,region,scale_m,window_id):
    ref=reference
    sam=predicted[roi_id]
    ref_in=ref.loc[ref.geometry.centroid.within(region)]
    sam_in=sam.loc[sam.geometry.centroid.within(region)]
    nref,nsam=len(ref_in),len(sam_in)
    ref_geom=union_clipped(ref.loc[ref.geometry.intersects(region)].geometry,region)
    sam_geom=union_clipped(sam.loc[sam.geometry.intersects(region)].geometry,region)
    common=ref_geom.intersection(sam_geom).area
    ref_area,sam_area=ref_geom.area,sam_geom.area
    region_area=region.area
    ref_only=(ref_area-common)/region_area*100
    sam_only=(sam_area-common)/region_area*100
    signed=(sam_area-ref_area)/region_area*100
    gross=ref_only+sam_only
    ref_sizes=ref_in.geometry.area.to_numpy()
    sam_sizes=sam_in.geometry.area.to_numpy()
    can_union=ref_area+sam_area-common
    return {
        "ROI":roi_id,"scale_m":scale_m,"window_id":window_id,
        "window_ha":region_area/1e4,
        "ref_n":nref,"sam_n":nsam,
        "count_error":nsam-nref,
        "density_bias_pct":100*(nsam-nref)/nref if nref>0 else np.nan,
        "ref_mean_area_m2":float(np.mean(ref_sizes)) if nref else np.nan,
        "sam_mean_area_m2":float(np.mean(sam_sizes)) if nsam else np.nan,
        "ref_median_area_m2":float(np.median(ref_sizes)) if nref else np.nan,
        "ref_cv_area":float(np.std(ref_sizes,ddof=1)/np.mean(ref_sizes)) if nref>=3 else np.nan,
        "ref_canopy_pct":ref_area/region_area*100,
        "sam_canopy_pct":sam_area/region_area*100,
        "cover_bias_pp":signed,
        "abs_cover_error_pp":abs(signed),
        "sam_only_pp":sam_only,
        "ref_only_pp":ref_only,
        "gross_spatial_error_pp":gross,
        "cancellation_fraction":1-abs(signed)/gross if gross>0 else np.nan,
        "canopy_iou":common/can_union if can_union>0 else np.nan
    }

metric_rows=[]
for roi in ROIS:
    print("ROI",roi,"running window metrics...",flush=True)
    region=roi_gdf.geometry.iloc[roi]
    metric_rows.append(metric_pair(roi,region,0,"full_ROI"))
    for row in WINDOWS_GDF.loc[WINDOWS_GDF.ROI==roi].itertuples():
        metric_rows.append(metric_pair(roi,row.geometry,int(row.scale_m),int(row.window_id)))
METRICS=pd.DataFrame(metric_rows)
display(METRICS.loc[METRICS.scale_m.eq(0),
                    ["ROI","ref_n","sam_n","cover_bias_pp","canopy_iou","gross_spatial_error_pp","cancellation_fraction"]].round(3))
METRICS.to_csv(OUT/"04_window_ecological_metrics.csv",index=False)
print("Window rows:",len(METRICS),"saved:",OUT/"04_window_ecological_metrics.csv")

## 05 — Main figure: how ecological error changes with spatial scale

Summarize errors **within each ROI first** and then show equal-ROI mean and spread across six ROIs. At any scale, a ROI contributes only if a full window exists. **This does not estimate causal scaling or a precise breakpoint**: small windows and whole-ROI estimates cover different footprints, and ROI sample size is six.

Make two primary figures: (A) mean absolute canopy-cover error versus window width with ROI trajectories and (B) gross disagreement versus net bias, demonstrating error cancellation. These are publication candidates after validation with the manuscript's final segmentation output.

In [ ]:
# 05 — Per-ROI scale summaries, weighted equally across independent ROIs
SCALE_ROI=(METRICS.assign(scale_label=METRICS.scale_m)
           .groupby(["ROI","scale_label"])
           .agg(n_windows=("window_id","size"),
                mean_abs_cover_pp=("abs_cover_error_pp","mean"),
                mean_abs_count_error=("count_error",lambda x:float(np.mean(np.abs(x)))),
                mean_gross_disagreement_pp=("gross_spatial_error_pp","mean"),
                mean_cancellation_fraction=("cancellation_fraction","mean"),
                mean_canopy_iou=("canopy_iou","mean"),
                mean_signed_cover_pp=("cover_bias_pp","mean"))
           .reset_index())
pooled=SCALE_ROI.groupby("scale_label").agg(
    n_rois=("ROI","nunique"),
    mean_abs_cover_pp=("mean_abs_cover_pp","mean"),
    sd_between_roi=("mean_abs_cover_pp","std"),
    mean_iou=("mean_canopy_iou","mean"),
    mean_gross_disagreement_pp=("mean_gross_disagreement_pp","mean")).reset_index()
pooled=pooled.sort_values("scale_label")
display(pooled.round(3))
SCALE_ROI.to_csv(OUT/"04_scale_by_roi.csv",index=False)
pooled.to_csv(OUT/"04_scale_summary_equal_roi.csv",index=False)

fig,axes=plt.subplots(1,2,figsize=(13,5),constrained_layout=True)
for roi,g in SCALE_ROI.groupby("ROI"):
    g=g.sort_values("scale_label")
    axes[0].plot(g.scale_label.replace({0:160}),g.mean_abs_cover_pp,
                 marker="o",alpha=.65,label=f"ROI {roi}")
axes[0].set(xlabel="Window width (m; whole ROI displayed at 160m)",
            ylabel="Mean absolute canopy-cover error (pp)",
            title="Ecological cover error versus aggregation scale")
axes[0].set_xticks([10,20,40,80,160],["10","20","40","80","Whole ROI"])
axes[0].legend(fontsize=8,ncol=2)
axes[0].grid(alpha=.2)
axes[1].plot(pooled.scale_label.replace({0:160}),
             pooled.mean_gross_disagreement_pp,marker="s",label="Gross spatial disagreement")
axes[1].plot(pooled.scale_label.replace({0:160}),
             pooled.mean_abs_cover_pp,marker="o",label="Net absolute cover error")
axes[1].set(xlabel="Window width (m; whole ROI shown at 160m)",
            ylabel="Area fraction (% ROI or pp)",
            title="Spatial error cancellation across scales")
axes[1].set_xticks([10,20,40,80,160],["10","20","40","80","Whole ROI"])
axes[1].legend();axes[1].grid(alpha=.2)
fig.savefig(OUT/"04_scale_dependence_and_error_cancellation.png",dpi=200,bbox_inches="tight")
plt.show()
print("WARNING: whole ROI is not a 160m square; position is categorical for visualization.")

## 06 — Structural predictors: reference crown size, density, heterogeneity and crowding

Reference-based predictors help explain segmentation errors **without circularly defining structure from SAM predictions**. For each window, calculate:

- Crown density per hectare (reference centroid crowns).
- Median crown area and coefficient of variation (≥3 crowns for CV).
- Mean nearest-neighbour distance between reference crown centroids, calculated from the **whole ROI's** crown set to avoid spuriously large distances at window boundaries.
- Reference union canopy cover and density as structural indicators.

Empty or sparse windows remain in the cover analysis but are excluded when their crown-based predictors are undefined. As each window can overlap other scales, compare structural relationships **within one fixed scale** first (e.g., 20 or 40 m).

In [ ]:
# 06 — Reference-only crowding and heterogeneity predictors
crowd_maps={}
for roi in ROIS:
    region=roi_gdf.geometry.iloc[roi]
    selected=reference.loc[reference.geometry.centroid.within(region)]
    xy=np.array([[p.x,p.y] for p in selected.geometry.centroid],dtype=float).reshape(-1,2)
    if len(xy)>=2:
        tree=cKDTree(xy);d,_=tree.query(xy,k=2)
        crowd_maps[roi]=(selected.index.to_numpy(),d[:,1])
    else:
        crowd_maps[roi]=(selected.index.to_numpy(),np.full(len(xy),np.nan))
crowd_series={roi:pd.Series(dist,index=ind) for roi,(ind,dist) in crowd_maps.items()}
def local_crowding(roi,region):
    selected=reference.loc[reference.geometry.centroid.within(region)]
    vals=crowd_series[roi].reindex(selected.index).dropna()
    return float(vals.mean()) if len(vals) else np.nan
crowd_rows=[]
for row in METRICS.itertuples():
    roi=int(row.ROI)
    if row.scale_m==0:geometry=roi_gdf.geometry.iloc[roi]
    else:
        geometry=WINDOWS_GDF.loc[
            (WINDOWS_GDF.ROI==roi)&(WINDOWS_GDF.scale_m==row.scale_m)&
            (WINDOWS_GDF.window_id==row.window_id),"geometry"].iloc[0]
    crowd_rows.append(local_crowding(roi,geometry))
METRICS["ref_mean_NN_distance_m"]=crowd_rows
METRICS["ref_density_ha"]=METRICS.ref_n/METRICS.window_ha
METRICS.to_csv(OUT/"04_window_ecological_structure_metrics.csv",index=False)
show=METRICS.loc[METRICS.scale_m.eq(40),[
    "ROI","ref_n","ref_density_ha","ref_median_area_m2","ref_cv_area",
    "ref_mean_NN_distance_m","ref_canopy_pct","abs_cover_error_pp"]]
print("=== Reference forest structure and cover error at 40 m ===")
display(show.head(20).round(2))

## 07 — Forest structure versus segmentation performance

Choose a **single fixed spatial scale** (40 m by default) for a descriptive first analysis. Display between- and within-ROI patterns for reference density, median crown area, size heterogeneity, mean neighbour distance and reference canopy cover versus absolute cover error and canopy IoU.

Compute **within-ROI demeaned Spearman-like descriptives** only as exploratory correlations, and show ROI-specific scatters. Neither windows nor individual crowns are treated as independent stands; no naive p-values are reported.

Be particularly careful with the reference canopy cover predictor: its association with cover-error metrics has potential **mathematical coupling** because reference cover appears in both quantities. Use density/crown-size/crowding relationships as the primary structural comparisons.

In [ ]:
# 07 — Descriptive structural dependence, fixed scale and ROI-centred
FIXED_SCALE=40
D=METRICS.loc[METRICS.scale_m==FIXED_SCALE].copy()
predictors=[
    ("ref_density_ha","Reference density (crowns/ha)"),
    ("ref_median_area_m2","Median reference crown area (m²)"),
    ("ref_cv_area","Reference crown-size CV"),
    ("ref_mean_NN_distance_m","Mean nearest-neighbour spacing (m)"),
    ("ref_canopy_pct","Reference canopy cover (%)")]
fig,axes=plt.subplots(2,3,figsize=(14,8),constrained_layout=True)
association=[]
for ax,(column,label) in zip(axes.flat,predictors):
    subset=D.dropna(subset=[column,"abs_cover_error_pp"])
    for roi,g in subset.groupby("ROI"):
        ax.scatter(g[column],g.abs_cover_error_pp,s=25,alpha=.65,label=f"ROI {roi}")
    ax.set(xlabel=label,ylabel="Abs canopy-cover error (pp)")
    ax.grid(alpha=.15)
    if len(subset)>=5 and subset[column].nunique()>2:
        centred_x=subset[column]-subset.groupby("ROI")[column].transform("mean")
        centred_y=subset.abs_cover_error_pp-subset.groupby("ROI").abs_cover_error_pp.transform("mean")
        rho,_=spearmanr(centred_x,centred_y)
    else:rho=np.nan
    association.append({"scale_m":FIXED_SCALE,"predictor":column,
                        "n_windows":len(subset),"n_rois":subset.ROI.nunique(),
                        "descriptive_rank_corr_after_ROI_demean":rho})
axes.flat[5].axis("off")
axes.flat[0].legend(fontsize=8,ncol=2)
fig.suptitle(f"Exploratory forest-structure associations, {FIXED_SCALE} × {FIXED_SCALE} m windows")
fig.savefig(OUT/"04_structure_dependence_40m.png",dpi=190,bbox_inches="tight")
plt.show()
ASSOCIATIONS=pd.DataFrame(association)
display(ASSOCIATIONS.round(3))
ASSOCIATIONS.to_csv(OUT/"04_exploratory_structure_associations.csv",index=False)
print("These are descriptive within-ROI associations, not independent stand-level inference.")

## 08 — Manuscript integration, quality controls and hand-off

**Proposed new manuscript subsection:** *Propagation of crown delineation errors into ecological metrics across spatial scales and forest structures.*

**Figures:** 04_scale_dependence_and_error_cancellation.png; 04_structure_dependence_40m.png; selected ROI 4 error map from Notebook 02.

**Minimum QA before paper use**
1. Check number and coverage of full windows per ROI at each scale. A drop in error can reflect changing sampled areas, not aggregation alone. A follow-up sensitivity analysis should use nested, common footprints and alternate grid origins.
2. Relate each ROI to the **correct field plot/stand age**; Nadine's manuscript has seven stands, but our new data have six ROIs.
3. Interpret relative density errors only where reference counts are nonzero. Investigate annotation completeness, particularly in dense young stands.
4. Confirm whether the manuscript's SAM-frst polygons (RGB/height fusion + SLIC) can be processed identically; the current results are based on the frozen RGB-only SAM1 + NMS baseline, *not* the manuscript's final SAM-frst workflow.
5. Avoid interpreting window-level sample size as independent replication. Six ROIs are the primary sampling strata; any inferential analysis requires spatially honest uncertainty.

**Main scientific framing:** Crown-level F1, area-based canopy agreement and aggregate ecological metrics are complementary. The scale at which a model is fit for ecological purpose is not determined by crown IoU alone.

In [ ]:
# 08 — Export concise tables for manuscript discussion
print("Whole-ROI metrics:")
display(METRICS.loc[METRICS.scale_m.eq(0),[
    "ROI","ref_n","sam_n","density_bias_pct","cover_bias_pp",
    "canopy_iou","gross_spatial_error_pp","cancellation_fraction"]].round(3))
print("Scale summaries:")
display(pooled.round(3))
print("Saved outputs:",OUT)
print("No SAM inference. Original crown polygons and annotations unchanged.")